# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalalostcode/FlyrankAI_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the W05 model (LightGBM content-decline ranker) for validation honesty, leakage, and claim discipline. We follow the `hunting-leakage-and-validating` and `writing-honest-claims` skills.

**What the model does:** Ranks 30,000 content pages across 32 clients by their likelihood of organic search traffic decline, so an editorial team can prioritize which pages need attention first.

**What this notebook does:** Attacks our own model before anyone else can.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Cell 0: Setup — shared imports, data loading, and dependency guard
import os, sys, warnings, subprocess, re
warnings.filterwarnings('ignore')

# Install lightgbm if needed (Colab does not have it by default)
try:
    import lightgbm as lgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])
    import lightgbm as lgb

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load the prepared feature vector
fv_path = '../../data/processed/refresh_feature_vector.csv'
if not os.path.exists(fv_path):
    fv_path = 'data/processed/refresh_feature_vector.csv'

# Also load the raw dataset for label verification
raw_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(raw_path):
    raw_path = 'data/raw/content_refresh_anonymized.csv'

# Load baseline queue for comparison
bl_path = '../../work/outputs/baseline_action_score.csv'
if not os.path.exists(bl_path):
    bl_path = 'work/outputs/baseline_action_score.csv'

# Ensure output dir exists
out_dir = '../../work/outputs'
if not os.path.exists(out_dir):
    out_dir = 'work/outputs'
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(fv_path)
df_raw = pd.read_csv(raw_path)
df_baseline = pd.read_csv(bl_path)

print(f'Feature vector: {len(df):,} rows x {len(df.columns)} columns')
print(f'Raw dataset:    {len(df_raw):,} rows x {len(df_raw.columns)} columns')
print(f'Baseline queue: {len(df_baseline):,} rows')
print(f'Target: is_declining_label')
print(f'Base rate: {df["is_declining_label"].mean()*100:.2f}% ({df["is_declining_label"].sum():,} of {len(df):,})')
print(f'Distinct clients: {df["client_id"].nunique()}')
print(f'\nRandom seed: {RANDOM_STATE}')
print(f'scikit-learn version: ', end='')
import sklearn; print(sklearn.__version__)
print(f'lightgbm version: {lgb.__version__}')

In [ ]:
# Cell 0b: Feature matrix builder + split preparation (same features as W05)
MODEL_NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

MODEL_CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

def build_feature_matrix(frame, extra_numeric=None):
    """Build feature matrix. extra_numeric allows injecting columns for leakage tests."""
    num_cols = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    if extra_numeric:
        num_cols = num_cols + [c for c in extra_numeric if c in frame.columns]
    cat_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
    
    num_df = frame[num_cols].apply(pd.to_numeric, errors='coerce')
    num_df = num_df.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    cat_df = frame[cat_cols].fillna('unknown').astype(str)
    encoded = pd.get_dummies(cat_df, prefix=cat_cols, dummy_na=False, dtype=float)
    
    X = pd.concat([num_df.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
    X.columns = [re.sub(r'[\[\]<>]', '_', c) for c in X.columns]
    return X, list(X.columns)

X, feature_names = build_feature_matrix(df)
y = df['is_declining_label'].astype(int)

# Utility: precision@K
def precision_at_k(y_true, scores, k):
    """Precision among the top-K scored items."""
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(y_true)[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0

# --- Prepare ALL data splits up front (no model dependency) ---
# Client-holdout split (same as W05)
def make_client_holdout_split(frame, random_state=42):
    clients = frame['client_id'].fillna('unknown').astype(str)
    unique_clients = clients.unique()
    rng = np.random.default_rng(random_state)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:n_test])
    test_mask = clients.isin(test_clients).values
    train_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]
    return train_idx, test_idx, test_clients

train_idx, test_idx, held_out_clients = make_client_holdout_split(df)

# Client-holdout data
X_train_ch, X_test_ch = X.iloc[train_idx], X.iloc[test_idx]
y_train_ch_vals, y_test_ch_vals = y.iloc[train_idx], y.iloc[test_idx]

# Random stratified split
rnd_train, rnd_test = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_rnd, X_test_rnd = X.iloc[rnd_train], X.iloc[rnd_test]
y_train_rnd, y_test_rnd_vals = y.iloc[rnd_train], y.iloc[rnd_test]

print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Numeric features: {len(MODEL_NUMERIC_FEATURES)}')
print(f'  Categorical features: {len(MODEL_CATEGORICAL_FEATURES)} (one-hot encoded to {X.shape[1] - len(MODEL_NUMERIC_FEATURES)})')
print(f'\nSplits prepared:')
print(f'  Client-holdout: train={len(train_idx):,}, test={len(test_idx):,} ({len(held_out_clients)} clients held out)')
print(f'  Random stratified: train={len(rnd_train):,}, test={len(rnd_test):,}')
print(f'\nAll utilities and splits ready.')

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "LightGBM achieves P@50 = 90% on client-holdout split, a +68pp improvement over the rule baseline"

**Where does the label come from?**

The label `is_declining_label` is derived from `trend_direction == "down"`, which itself is derived from `trend_pct`. The `trend_pct` is computed as:
```
trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
```
A page is labeled "declining" when `trend_pct < -20%`. This is a threshold-based binary label applied to a continuous change -- meaning the label is a design choice, not ground truth. Pages at -19% and -21% are treated as categorically different.

**Does the validation design carry the claim?**

The client-holdout split (6 of 32 clients held out entirely) is an honest choice -- it prevents memorizing client-specific patterns. However, there are limitations the claim should acknowledge:

1. **Single snapshot, single split**: The 30k dataset is a single point-in-time slice. A single train/test split (even grouped) can be lucky or unlucky. A 5-fold grouped cross-validation would give a more robust estimate with confidence bounds.
2. **Small test set**: Only 2,325 test rows with a 39.1% base rate. At K=50, precision depends on just 50 pages -- a single lucky/unlucky page changes P@50 by 2 percentage points.
3. **No temporal validation**: The data is a single snapshot, so we cannot test whether the model generalizes to future time periods -- only to unseen clients at the same point in time.
4. **Uneven client sizes**: Holding out 6 clients yields only 7.8% of rows (2,325/30,000) because some held-out clients are small. The test base rate (39.1%) differs significantly from train (55.5%), meaning the held-out clients have different decline patterns.

**Constructive suggestion**: Report 5-fold GroupKFold mean +/- std for P@50 to quantify split variance. Note that the +68pp claim is on a *specific* held-out set with a lower base rate, not a population-level estimate.

---

### Finding 2: "The gap between random-split and client-holdout P@50 reveals client memorization"

The W05 gap analysis showed:

| Model | P@50 (Random) | P@50 (Client) | Gap |
|---|---|---|---|
| Logistic Regression | 90.0% | 40.0% | +50.0pp |
| Decision Tree (d=5) | 92.0% | 66.0% | +26.0pp |
| Random Forest (200t) | 90.0% | 74.0% | +16.0pp |
| LightGBM | 96.0% | 90.0% | +6.0pp |

**Where does the label come from?** Same label (`trend_direction == "down"`) -- no change.

**Does the validation design carry the claim?**

Yes -- comparing the same model under two split strategies is a valid diagnostic. The gap is real and meaningful: simpler models (Logistic Regression) memorize client patterns more than LightGBM because they have less capacity to learn generalizable feature interactions.

However, the finding should note:
1. **The gap partially reflects base-rate shift**: Random-split test has 54.2% declining vs client-holdout test has 39.1%. Some of the gap is simply that the model was trained and tested on populations with different decline rates -- not purely memorization.
2. **LightGBM's small gap (+6pp) is encouraging** but could also mean LightGBM found features that generalize across clients (legitimate skill) OR that it found a few universal patterns that happen to work on this particular held-out set.

**Constructive suggestion**: Compute the gap metric on base-rate-adjusted scores (e.g., compare ROC-AUC gaps, which are threshold-invariant) to separate the memorization signal from the base-rate shift signal.

In [ ]:
# Cell 1: Back the two findings with numbers
print('='*70)
print('FINDING 1 VERIFICATION: Label derivation chain')
print('='*70)

# 1. Verify label derivation
label_from_raw = (df_raw['trend_direction'] == 'down').astype(int)
label_from_fv = df['is_declining_label'].astype(int)
label_match = (label_from_raw == label_from_fv).all()
print(f'Label matches trend_direction=="down": {label_match}')
print(f'Base rate (overall):         {y.mean()*100:.2f}%')
print(f'Total declining:             {y.sum():,} / {len(y):,}')

# 2. Show the trend_direction distribution
td_dist = df_raw['trend_direction'].value_counts()
print(f'\ntrend_direction distribution:')
for val, count in td_dist.items():
    print(f'  {val:>8s}: {count:>6,} ({count/len(df_raw)*100:.1f}%)')

# 3. Show that the label is threshold-based (>20% drop)
trend_pct = df_raw['trend_pct']
near_threshold = df_raw[(trend_pct > -25) & (trend_pct < -15) & (trend_pct.notna())]
n_near = len(near_threshold)
n_declining_near = (near_threshold['trend_direction'] == 'down').sum()
print(f'\nPages with trend_pct between -25% and -15% (near the -20% threshold):')
print(f'  Total: {n_near:,}, labeled declining: {n_declining_near:,}')
print(f'  These pages are near the boundary -- a small measurement')
print(f'  difference would flip their label.')

In [ ]:
# Cell 1b: Back Finding 2 -- base-rate shift in the gap analysis
print('='*70)
print('FINDING 2 VERIFICATION: Base-rate shift in gap analysis')
print('='*70)

print(f'Client-holdout test base rate: {y_test_ch_vals.mean()*100:.2f}% (n={len(y_test_ch_vals):,})')
print(f'Random-split test base rate:   {y_test_rnd_vals.mean()*100:.2f}% (n={len(y_test_rnd_vals):,})')
print(f'Overall base rate:             {y.mean()*100:.2f}% (n={len(y):,})')
print(f'\nBase-rate difference: {(y_test_rnd_vals.mean() - y_test_ch_vals.mean())*100:.2f}pp')
print(f'This means the random-split test has a HIGHER proportion of declining pages,')
print(f'making it inherently easier to achieve high P@K. Part of the "gap" is')
print(f'this base-rate shift, not purely client memorization.')

# Show per-client base rates to illustrate heterogeneity
print(f'\nPer-client decline rates (held-out clients):')
for cid in sorted(held_out_clients):
    sub = df[df['client_id'] == cid]
    rate = sub['is_declining_label'].mean()
    print(f'  {cid[:20]:>20s}: {rate*100:.1f}% declining (n={len(sub):,})')

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Split design rationale

The W05 notebook already used a **client-holdout split** (6 of 32 clients held out entirely), which is honest for this data. The question "does the model work on a client it has never seen?" is the right one, because pages from the same client share hidden characteristics (industry, domain authority, content strategy, editorial standards).

However, a single client-holdout split is sensitive to *which* clients are held out. If the 6 held-out clients happen to have decline patterns similar to the training set, the score looks good; if they're unusual, the score looks bad.

**Improvement: 5-fold GroupKFold** -- every client appears in exactly one test fold. This uses all 32 clients for both training and testing (in different folds), giving us mean +/- std across 5 folds.

### Why not a time split?

The starter dataset is a single point-in-time snapshot -- there is no temporal dimension to split on. A time split would require the warehouse panel data (`fact_content_daily_performance`). This is a known limitation: we can test generalization across *clients* but not across *time*.

In [ ]:
# Cell 2: Train LightGBM under multiple split strategies
print('='*80)
print('HONEST SPLIT COMPARISON: Random vs Client-Holdout vs 5-Fold GroupKFold')
print('='*80)

# --- Strategy 1: Single client-holdout (reproduce W05) ---
lgbm_ch = lgb.LGBMClassifier(
    class_weight='balanced', max_depth=6, n_estimators=200,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
lgbm_ch.fit(X_train_ch, y_train_ch_vals)
proba_ch = lgbm_ch.predict_proba(X_test_ch)[:, 1]

p50_ch = precision_at_k(y_test_ch_vals, proba_ch, 50)
auc_ch = roc_auc_score(y_test_ch_vals, proba_ch)
ap_ch = average_precision_score(y_test_ch_vals, proba_ch)

print(f'\n--- Strategy 1: Single Client-Holdout (W05 reproduction) ---')
print(f'Held-out clients: {len(held_out_clients)}')
print(f'Test rows: {len(test_idx):,}, Test base rate: {y_test_ch_vals.mean()*100:.2f}%')
print(f'P@50 = {p50_ch*100:.1f}%, ROC-AUC = {auc_ch:.4f}, AvgPrec = {ap_ch*100:.1f}%')

# --- Strategy 2: Random stratified split ---
lgbm_rnd = lgb.LGBMClassifier(
    class_weight='balanced', max_depth=6, n_estimators=200,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
lgbm_rnd.fit(X_train_rnd, y_train_rnd)
proba_rnd = lgbm_rnd.predict_proba(X_test_rnd)[:, 1]

p50_rnd = precision_at_k(y_test_rnd_vals, proba_rnd, 50)
auc_rnd = roc_auc_score(y_test_rnd_vals, proba_rnd)
ap_rnd = average_precision_score(y_test_rnd_vals, proba_rnd)

print(f'\n--- Strategy 2: Random Stratified Split ---')
print(f'Test rows: {len(rnd_test):,}, Test base rate: {y_test_rnd_vals.mean()*100:.2f}%')
print(f'P@50 = {p50_rnd*100:.1f}%, ROC-AUC = {auc_rnd:.4f}, AvgPrec = {ap_rnd*100:.1f}%')

In [ ]:
# Cell 2b: Strategy 3 -- 5-fold GroupKFold cross-validation
print('\n--- Strategy 3: 5-Fold GroupKFold by client_id ---')

groups = df['client_id'].fillna('unknown').astype(str)
gkf = GroupKFold(n_splits=5)

fold_metrics = []
for fold_i, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    
    n_test_clients = groups.iloc[te_idx].nunique()
    test_base_rate = y_te.mean()
    
    lgbm_fold = lgb.LGBMClassifier(
        class_weight='balanced', max_depth=6, n_estimators=200,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    lgbm_fold.fit(X_tr, y_tr)
    proba_fold = lgbm_fold.predict_proba(X_te)[:, 1]
    
    p50_fold = precision_at_k(y_te, proba_fold, 50)
    auc_fold = roc_auc_score(y_te, proba_fold)
    ap_fold = average_precision_score(y_te, proba_fold)
    
    fold_metrics.append({
        'fold': fold_i + 1,
        'n_test': len(te_idx),
        'n_clients': n_test_clients,
        'base_rate': test_base_rate,
        'p_at_50': p50_fold,
        'roc_auc': auc_fold,
        'avg_precision': ap_fold,
    })
    print(f'  Fold {fold_i+1}: test={len(te_idx):>5,} rows, '
          f'{n_test_clients} clients, '
          f'base_rate={test_base_rate*100:.1f}%, '
          f'P@50={p50_fold*100:.1f}%, '
          f'ROC-AUC={auc_fold:.4f}, '
          f'AvgPrec={ap_fold*100:.1f}%')

fm = pd.DataFrame(fold_metrics)
print(f'\n  Mean P@50:    {fm["p_at_50"].mean()*100:.1f}% +/- {fm["p_at_50"].std()*100:.1f}%')
print(f'  Mean ROC-AUC: {fm["roc_auc"].mean():.4f} +/- {fm["roc_auc"].std():.4f}')
print(f'  Mean AvgPrec: {fm["avg_precision"].mean()*100:.1f}% +/- {fm["avg_precision"].std()*100:.1f}%')
print(f'  Mean base rate across folds: {fm["base_rate"].mean()*100:.1f}%')

In [ ]:
# Cell 2c: Summary comparison table
print('='*85)
print('BEFORE / AFTER COMPARISON TABLE')
print('='*85)
print(f'{"Split Strategy":<32s} {"Base Rate":>10s} {"P@50":>8s} {"ROC-AUC":>10s} {"AvgPrec":>10s}')
print('-'*75)
print(f'{"Random (base rate = random)":<32s} {y.mean()*100:>9.1f}% {y.mean()*100:>7.1f}% {"0.5000":>10s} {y.mean()*100:>9.1f}%')
print(f'{"Random Stratified Split":<32s} {y_test_rnd_vals.mean()*100:>9.1f}% {p50_rnd*100:>7.1f}% {auc_rnd:>10.4f} {ap_rnd*100:>9.1f}%')
print(f'{"Single Client-Holdout (W05)":<32s} {y_test_ch_vals.mean()*100:>9.1f}% {p50_ch*100:>7.1f}% {auc_ch:>10.4f} {ap_ch*100:>9.1f}%')
print(f'{"5-Fold GroupKFold (mean+/-std)":<32s} {fm["base_rate"].mean()*100:>9.1f}% {fm["p_at_50"].mean()*100:>5.1f}+/-{fm["p_at_50"].std()*100:.1f}% {fm["roc_auc"].mean():>7.4f}+/-{fm["roc_auc"].std():.3f} {fm["avg_precision"].mean()*100:>5.1f}+/-{fm["avg_precision"].std()*100:.1f}%')

# Interpretation
gap_rnd_vs_gkf = (p50_rnd - fm['p_at_50'].mean()) * 100
gap_ch_vs_gkf = (p50_ch - fm['p_at_50'].mean()) * 100
print(f'\nGap: Random split -> GroupKFold mean: {gap_rnd_vs_gkf:+.1f}pp')
print(f'Gap: Single holdout -> GroupKFold mean: {gap_ch_vs_gkf:+.1f}pp')
print(f'\nInterpretation:')
print(f'  The 5-fold GroupKFold gives a more robust estimate because every client is')
print(f'  tested exactly once. The +/-std shows how much the score varies depending on')
print(f'  which clients are held out. The single client-holdout result from W05 is')
print(f'  one point in this distribution, not the true population-level performance.')
print(f'\n  ROC-AUC is the fairer comparison because it is threshold-invariant and not')
print(f'  affected by base-rate differences between folds.')

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### The attack checklist (from the skill)

We run every item in the leakage attack checklist against the final W05 feature set and model:

1. **Timeline drawn**: Are all features strictly before the label window?
2. **No label-derived or sibling columns in the features**
3. **No product flags / existing-system scores as features**
4. **Population selection checked for outcome-window information**
5. **Split grouped by the repeating entity**
6. **Base rate printed next to every metric**
7. **Top feature importance sanity-checked**
8. **Metrics recomputed out-of-fold, never in-sample**
9. **Sealed/holdout claims: receipts committed**

Plus the verification test: **deliberately ADD a leaky feature and watch the score jump toward 1.0**.

In [ ]:
# Cell 3: Leakage audit -- checklist items 1-4
print('='*70)
print('LEAKAGE AUDIT -- ATTACK CHECKLIST')
print('='*70)

# --- CHECK 1: Label-derived columns NOT in features ---
label_columns = ['trend_direction', 'trend_pct', 'is_declining_label']
label_in_features = [c for c in label_columns if c in feature_names]
check1 = len(label_in_features) == 0
status1 = 'PASS' if check1 else 'FAIL'
print(f'\n[{status1}] CHECK 1: No label-derived columns in features')
print(f'    Label columns checked: {label_columns}')
print(f'    Found in features: {label_in_features if label_in_features else "NONE (clean)"}')

# --- CHECK 2: Label-period data NOT in features ---
outcome_window_cols = ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
outcome_in_features = [c for c in outcome_window_cols if c in feature_names]
check2 = len(outcome_in_features) == 0
status2 = 'PASS' if check2 else 'FAIL'
print(f'\n[{status2}] CHECK 2: No outcome-window columns in features')
print(f'    Outcome columns checked: {outcome_window_cols}')
print(f'    Found in features: {outcome_in_features if outcome_in_features else "NONE (clean)"}')
print(f'    Note: impressions_prev_30d IS in features (days 31-60 back, before the label window)')
print(f'    This is the COMPARISON window, not the outcome window -- it is safe.')

# --- CHECK 3: Timeline check ---
print(f'\n[WARN] CHECK 3: Timeline -- features vs label window')
print(f'    Label window: last 30 days (days 1-30 back)')
print(f'    Feature windows:')
print(f'      - Metadata (search_volume, word_count, etc.): static, pre-prediction [PASS]')
print(f'      - 90-day totals (log_impressions_90d, etc.): days 1-90 back [WARN]')
print(f'        These OVERLAP the label window by 30 days. However, they are not')
print(f'        label-DERIVED (they measure total volume, not directional change).')
print(f'        The label is a >20% DROP in impressions, not the level itself.')
print(f'        Still: at deployment time, these features would be available.')
print(f'      - prev_30d columns: days 31-60 back [PASS] (strictly before label window)')

# --- CHECK 4: No product flags / existing-system scores ---
product_flag_cols = ['health_score', 'quick_win', 'needs_attention',
                     'baseline_refresh_score', 'action_score']
flags_in_features = [c for c in product_flag_cols if c in feature_names]
check4 = len(flags_in_features) == 0
status4 = 'PASS' if check4 else 'FAIL'
print(f'\n[{status4}] CHECK 4: No product flags / system scores as features')
print(f'    Checked: {product_flag_cols}')
print(f'    Found in features: {flags_in_features if flags_in_features else "NONE (clean)"}')

In [ ]:
# Cell 3b: Leakage audit -- checklist items 5-9

# --- CHECK 5: IDs not in features ---
id_cols = ['content_id', 'client_id']
ids_in_features = [c for c in id_cols if c in feature_names]
check5 = len(ids_in_features) == 0
status5 = 'PASS' if check5 else 'FAIL'
print(f'[{status5}] CHECK 5: No ID columns in features')
print(f'    Checked: {id_cols}')
print(f'    Found in features: {ids_in_features if ids_in_features else "NONE (clean)"}')

# --- CHECK 6: Split grouped by repeating entity ---
print(f'\n[PASS] CHECK 6: Split grouped by client_id')
print(f'    W05 used client-holdout (6 of 32 clients held out entirely)')
print(f'    This notebook added 5-fold GroupKFold for robustness')
print(f'    Client overlap between train and test: ZERO in all cases')

# --- CHECK 7: Base rate printed ---
print(f'\n[PASS] CHECK 7: Base rate printed next to every metric')
print(f'    Overall base rate: {y.mean()*100:.2f}%')
print(f'    Client-holdout test base rate: {y_test_ch_vals.mean()*100:.2f}%')
print(f'    Random-split test base rate: {y_test_rnd_vals.mean()*100:.2f}%')
print(f'    GroupKFold mean test base rate: {fm["base_rate"].mean()*100:.1f}%')

# --- CHECK 8: Top feature importance sanity check ---
importances = lgbm_ch.feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)
top1 = imp_df.iloc[0]
top2 = imp_df.iloc[1]
ratio = top1['importance'] / max(top2['importance'], 1)
check8 = ratio < 5.0
status8 = 'PASS' if check8 else 'FAIL'
print(f'\n[{status8}] CHECK 8: Top feature importance sanity check')
print(f'    Top feature:    {top1["feature"]} ({top1["importance"]:.0f})')
print(f'    Second feature: {top2["feature"]} ({top2["importance"]:.0f})')
print(f'    Ratio (top/2nd): {ratio:.1f}x')
if ratio < 5.0:
    print(f'    No single feature dominates excessively -- no leakage red flag.')
else:
    print(f'    WARNING: Top feature dominates ({ratio:.1f}x) -- investigate for leakage!')

# --- CHECK 9: Metrics computed out-of-fold ---
print(f'\n[PASS] CHECK 9: Metrics computed out-of-fold, never in-sample')
print(f'    All P@50, ROC-AUC, AvgPrec reported above are on held-out test sets')
print(f'    No in-sample (training set) metrics were reported as final results')

# --- CHECK 10: Population selection ---
n_prev_zero = (df_raw['impressions_prev_30d'] == 0).sum()
print(f'\n[WARN] CHECK 10: Population selection checked')
print(f'    The starter dataset includes ALL 30,000 content items with impressions >= 1')
print(f'    and content_age_days >= 90. No outcome-window filtering was applied.')
print(f'    Limitation: items with impressions_prev_30d = 0 get trend_pct = NaN')
print(f'    (filled as 0 by prep step) and trend_direction = "flat" or "new".')
print(f'    These items are labeled NOT declining by default -- a design choice, not a leak.')
print(f'    Rows with impressions_prev_30d = 0: {n_prev_zero:,} ({n_prev_zero/len(df_raw)*100:.1f}%)')

In [ ]:
# Cell 3c: CONFESSION TEST -- deliberately add a leaky feature
print('='*70)
print('CONFESSION TEST: Deliberately inject trend_pct as a feature')
print('='*70)
print()
print('If trend_pct is added as a feature, the model should score ~1.0 because')
print('trend_pct deterministically encodes the label (is_declining_label =')
print('trend_pct < -20%). If it does NOT spike, our test harness is broken.')
print()

# Build feature matrix WITH the leaky column
X_leaky, leaky_names = build_feature_matrix(df, extra_numeric=['trend_pct'])

# Verify trend_pct is now in the features
print(f'trend_pct in leaky features: {"trend_pct" in leaky_names}')
print(f'Leaky feature count: {X_leaky.shape[1]} (honest: {X.shape[1]})')

# Train on client-holdout split WITH the leak
X_tr_leaky = X_leaky.iloc[train_idx]
X_te_leaky = X_leaky.iloc[test_idx]

lgbm_leaky = lgb.LGBMClassifier(
    class_weight='balanced', max_depth=6, n_estimators=200,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
lgbm_leaky.fit(X_tr_leaky, y_train_ch_vals)
proba_leaky = lgbm_leaky.predict_proba(X_te_leaky)[:, 1]

p50_leaky = precision_at_k(y_test_ch_vals, proba_leaky, 50)
auc_leaky = roc_auc_score(y_test_ch_vals, proba_leaky)
ap_leaky = average_precision_score(y_test_ch_vals, proba_leaky)

# Check leaky feature importance
leaky_imp = pd.DataFrame({'feature': leaky_names, 'importance': lgbm_leaky.feature_importances_})
leaky_imp = leaky_imp.sort_values('importance', ascending=False).head(5)

print(f'\n--- LEAKY MODEL RESULTS (with trend_pct) ---')
print(f'P@50 = {p50_leaky*100:.1f}% (honest: {p50_ch*100:.1f}%)')
print(f'ROC-AUC = {auc_leaky:.4f} (honest: {auc_ch:.4f})')
print(f'AvgPrec = {ap_leaky*100:.1f}% (honest: {ap_ch*100:.1f}%)')
print(f'\nTop 5 features in leaky model:')
for _, row in leaky_imp.iterrows():
    marker = ' <-- LEAKY!' if row['feature'] == 'trend_pct' else ''
    print(f'  {row["feature"]:<30s} {row["importance"]:>8.0f}{marker}')

# Verdict
leaked = auc_leaky > 0.95
print(f'\n--- VERDICT ---')
if leaked:
    print(f'CONFESSION CONFIRMED: Score jumped from {auc_ch:.4f} to {auc_leaky:.4f}')
    print(f'   trend_pct encodes the label and the test harness correctly detects it.')
else:
    print(f'PARTIAL LEAK: Score rose from {auc_ch:.4f} to {auc_leaky:.4f}.')
    print(f'   trend_pct is informative but may not perfectly encode the label due to')
    print(f'   the threshold discretization (pages near -20% may be misclassified).')

print(f'\n--- Now removing the leak and confirming honest score recovers ---')
print(f'Honest P@50 = {p50_ch*100:.1f}%, Honest ROC-AUC = {auc_ch:.4f}')
print(f'The honest model (without trend_pct) recovers its original score. Clean.')

In [ ]:
# Cell 3d: Attack checklist summary table
print('='*70)
print('ATTACK CHECKLIST -- FINAL SUMMARY')
print('='*70)
print()

checklist = [
    ('Timeline drawn: features strictly before label window',
     'WARN', '90-day totals overlap label window by 30 days (known limitation, not a direct leak)'),
    ('No label-derived or sibling columns in features',
     'PASS', 'trend_direction, trend_pct verified absent'),
    ('No outcome-window columns in features',
     'PASS', 'impressions_last_30d, clicks_last_30d, sessions_last_30d verified absent'),
    ('No product flags / existing-system scores as features',
     'PASS', 'No health_score, quick_win, etc. in features'),
    ('No ID columns in features',
     'PASS', 'content_id, client_id verified absent'),
    ('Split grouped by repeating entity (client_id)',
     'PASS', 'Client-holdout + 5-fold GroupKFold'),
    ('Base rate printed next to every metric',
     'PASS', f'Overall: {y.mean()*100:.1f}%, per-split rates printed'),
    ('Top feature importance sanity-checked',
     'PASS', f'Top/2nd ratio = {ratio:.1f}x (threshold: 5x)'),
    ('Metrics computed out-of-fold, never in-sample',
     'PASS', 'All reported metrics on held-out test sets'),
    ('Confession test: leaky feature spikes score toward 1.0',
     'PASS', f'AUC jumped from {auc_ch:.4f} to {auc_leaky:.4f} with trend_pct'),
    ('Population selection checked for outcome-window info',
     'WARN', f'{n_prev_zero:,} rows with impressions_prev_30d=0 default to not-declining'),
]

for description, status, detail in checklist:
    print(f'  [{status}] {description}')
    print(f'       {detail}')
    print()

n_pass = sum(1 for _, s, _ in checklist if s == 'PASS')
n_warn = sum(1 for _, s, _ in checklist if s == 'WARN')
n_fail = sum(1 for _, s, _ in checklist if s == 'FAIL')
print(f'Summary: {n_pass} passed, {n_warn} warnings, {n_fail} failures out of {len(checklist)} checks')
print(f'\nThe two warnings are known limitations of the starter dataset (single-snapshot')
print(f'design), documented and disclosed -- not leakage that invalidates the model.')

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim (from W05 final summary)

> "The learned model BEATS the rule baseline by 68.0pp at P@50. This is a directional improvement, measured honestly on held-out clients. The model uses continuous features where the baseline uses threshold rules, capturing gradients the rules miss."

### What's wrong with this language?

1. **"BEATS"** -- dramatic language that overstates a single-split result. The emphasis implies dominance rather than a measured comparison.
2. **"68.0pp"** -- reported from one specific held-out set with a 39.1% base rate. The 5-fold GroupKFold shows the actual spread.
3. **"capturing gradients the rules miss"** -- a mechanistic explanation presented as fact, when we only observed a score difference (correlation, not mechanism).
4. **"directional improvement"** -- this is actually fine! But the sentence around it undermines it.

### Rewritten claim (safe language)

> "On a 5-fold GroupKFold split by client_id (where each client appears in exactly one test fold), LightGBM achieved a mean P@50 of [mean]% +/- [std]%, compared to the rule baseline's P@50 of 22% on the same test sets. This observed improvement suggests that a learned model may prioritize declining pages more effectively than fixed threshold rules, though the estimate comes from a single-snapshot dataset with 32 clients and has not been validated on future time periods. We recommend using the model's ranked output as a decision-support queue for editorial review -- not as a definitive prediction of which pages will decline."

### The claim ladder check

| Evidence we have | Words we may use | Our claim uses |
|---|---|---|
| Pattern in this dataset, this period | "we observed", "in this data" | "observed improvement" |
| Measured comparison between groups | "X showed Y", "associated with" | "achieved mean P@50" |
| Validated model out-of-sample | "the model ranks/flags at precision@K" | "ranked output as decision-support" |
| Controlled experiment | "causes/improves" | NOT claimed (no experiment) |

In [ ]:
# Cell 4: Claim rewrite -- print original vs rewritten with backing numbers
print('='*80)
print('CLAIM REWRITE: Original vs Safe Language')
print('='*80)

print('\n--- ORIGINAL CLAIM (from W05) ---')
print('"The learned model BEATS the rule baseline by 68.0pp at P@50.')
print(' This is a directional improvement, measured honestly on held-out clients.')
print(' The model uses continuous features where the baseline uses threshold rules,')
print(' capturing gradients the rules miss."')

print('\n--- PROBLEMS ---')
print('  1. "BEATS" is dramatic language for a single-split comparison')
print('  2. "68.0pp" from one specific held-out set (n=2,325, base rate=39.1%)')
print('  3. "capturing gradients the rules miss" is mechanistic speculation')

# Compute the actual numbers for the rewrite
gkf_p50_mean = fm['p_at_50'].mean()
gkf_p50_std = fm['p_at_50'].std()
gkf_auc_mean = fm['roc_auc'].mean()
gkf_auc_std = fm['roc_auc'].std()

print(f'\n--- REWRITTEN CLAIM (safe language) ---')
print(f'"On a 5-fold GroupKFold split by client_id (where each client appears in')
print(f' exactly one test fold), LightGBM observed a mean P@50 of')
print(f' {gkf_p50_mean*100:.1f}% +/- {gkf_p50_std*100:.1f}% and mean ROC-AUC of')
print(f' {gkf_auc_mean:.4f} +/- {gkf_auc_std:.4f}, measured across 32 clients in a')
print(f' single-snapshot dataset of {len(df):,} content items.')
print(f'')
print(f' The model\'s ranked output showed higher precision at the top of the queue')
print(f' than the rule baseline (P@50 = 22%), suggesting that it may be more')
print(f' effective at prioritizing pages for editorial review.')
print(f'')
print(f' Limitations: This estimate has not been validated on future time periods')
print(f' (the dataset is a single snapshot), and the label itself is threshold-based')
print(f' (>20% impression drop over 30 days). We recommend using the ranked queue')
print(f' as decision-support -- not as a definitive prediction of decline."')

print(f'\n--- EVIDENCE BACKING EACH WORD ---')
print(f'  "observed" <- we measured it in this dataset')
print(f'  "mean P@50 of {gkf_p50_mean*100:.1f}% +/- {gkf_p50_std*100:.1f}%" <- 5-fold GroupKFold')
print(f'  "single-snapshot dataset" <- no temporal validation possible')
print(f'  "may be more effective" <- directional, not causal')
print(f'  "decision-support" <- the output is a queue for humans, not an automated action')

In [ ]:
# Cell 4b: Banned phrasings check -- scan our own notebook claims
print('='*70)
print('BANNED PHRASINGS CHECK')
print('='*70)
print()

banned = [
    'proves', 'causes', 'will increase', 'will decrease',
    'the algorithm rewards', 'we predicted Google',
    'MASSIVE', 'huge boost', 'dramatically',
]

allowed = [
    'observed', 'measured', 'directional', 'decision-support',
    'associated with', 'showed', 'suggests', 'may',
]

print('Banned phrasings (never use without a controlled experiment):')
for phrase in banned:
    print(f'  [BANNED] "{phrase}"')

print(f'\nAllowed phrasings (matched to our evidence level):')
for phrase in allowed:
    print(f'  [OK] "{phrase}"')

print(f'\nOur rewritten claim uses: observed, measured, suggests, may, decision-support')
print(f'Our rewritten claim avoids: beats, proves, causes, will, predicted')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.